# AI-Powered Environmental Issue Detection for San Jose

**AI for Social Good prototype**

This notebook demonstrates how a multimodal AI system can transform unstructured environmental reports and images into structured records for human review and future analytics.

> **Scope:** This prototype demonstrates text extraction, image analysis, and schema-guided JSON output. It does not connect to live city systems, IoT sensors, real-time alerts, or official emergency-response workflows.


## 1. Problem

Residents may report trash, blocked storm drains, water-quality concerns, or pipe leaks through informal messages and images. These reports can contain useful details, but the information is often inconsistent or difficult to analyze.

For example, one resident may write, “There is a strong chemical smell near the creek,” while another may report, “The drain is full of trash after the rain.” City staff would need to manually interpret these reports before categorizing the issue, estimating urgency, and deciding which service may need to review it.

This project focuses on the first step of the workflow: converting unstructured community reports into consistent, machine-readable records that can support human review and future dashboards.


## 2. AI capability and project objective

The project uses two capabilities:

1. **Schema-guided text extraction:** Converts a resident’s written report into seven standardized fields.
2. **Multimodal image analysis:** Uses an uploaded image and a structured prompt to describe a possible visible environmental issue.

The original prompt—`Extract water issue info.`—could produce a paragraph instead of predictable JSON. The revised prompt defines the required fields, allowed urgency values, and output format.


In [ ]:
!pip install -q openai

In [ ]:
import base64
import mimetypes

from openai import OpenAI
from google.colab import userdata, files

# Add OPENAI_API_KEY to Google Colab Secrets
client = OpenAI(
    api_key=userdata.get("OPENAI_API_KEY")
)

## 3. Structured report schema

Every report should contain the same seven fields. The output is intended for preliminary organization and triage—not for independently determining whether water is safe or whether an emergency response is required.


In [ ]:
REQUIRED_FIELDS = {
    "location",
    "issue_type",
    "contaminants_observed",
    "urgency",
    "affected_entities",
    "department",
    "resident_language",
}
VALID_URGENCY = {"LOW", "MEDIUM", "HIGH", "CRITICAL"}

schema_prompt = """
You organize environmental issue reports for preliminary human review.
Return ONLY valid JSON with exactly these seven fields:
{
  "location": string,
  "issue_type": string,
  "contaminants_observed": array of strings,
  "urgency": "LOW", "MEDIUM", "HIGH", or "CRITICAL",
  "affected_entities": array of strings,
  "department": string,
  "resident_language": string
}

Rules:
- Extract only information supported by the report or image.
- If a value is unknown, use "unknown" or an empty list.
- Do not claim that water is safe or unsafe based only on an image.
- Use a preliminary department suggestion, not a final official referral.
- Do not include markdown or explanatory text.
"""

def validate_report(report):
    """Return validation errors for a model-generated report."""
    errors = []
    missing = REQUIRED_FIELDS - set(report.keys())
    if missing:
        errors.append(f"Missing fields: {', '.join(sorted(missing))}")
    if set(report.keys()) != REQUIRED_FIELDS:
        extra = set(report.keys()) - REQUIRED_FIELDS
        if extra:
            errors.append(f"Unexpected fields: {', '.join(sorted(extra))}")
    if report.get("urgency") not in VALID_URGENCY:
        errors.append("urgency must be LOW, MEDIUM, HIGH, or CRITICAL")
    for field in ("contaminants_observed", "affected_entities"):
        if not isinstance(report.get(field), list):
            errors.append(f"{field} must be a list")
    return errors

def parse_json_response(raw_text):
    """Parse JSON and remove optional markdown fences returned by a model."""
    raw = raw_text.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    return json.loads(raw)

print("Schema and validation functions are ready.")


Schema and validation functions are ready.


In [ ]:
import json

MODEL_NAME = "gpt-4.1-mini"
RUN_LIVE_API = True


def extract_structured(message):
    response = client.responses.create(
        model=MODEL_NAME,
        input=[
            {
                "role": "system",
                "content": """
You organize environmental issue reports.

Return ONLY valid JSON with exactly these fields:
{
  "location": "",
  "issue_type": "",
  "contaminants_observed": [],
  "urgency": "LOW",
  "affected_entities": [],
  "department": "",
  "resident_language": ""
}

Use "unknown" when information is missing.
Do not include markdown or explanations.
"""
            },
            {
                "role": "user",
                "content": message
            }
        ]
    )

    raw_output = response.output_text.strip()

    if raw_output.startswith("```"):
        raw_output = raw_output.split("\n", 1)[1].rsplit("```", 1)[0].strip()

    report = json.loads(raw_output)

    # Optional validation if this function exists
    errors = validate_report(report) if "validate_report" in globals() else []

    return report, errors

## 4. Text-report demonstration

The following example shows how a free-form resident message can become a consistent structured record.


In [ ]:
sample_output = {
    "location": "Elm Street near the storm drain",
    "issue_type": "possible water contamination",
    "contaminants_observed": ["unknown"],
    "urgency": "HIGH",
    "affected_entities": ["creek", "residents"],
    "department": "Environmental Services",
    "resident_language": "English"
}

print("Example expected output — not a live API response:")
print(json.dumps(sample_output, indent=2))

Example expected output — not a live API response:
{
  "location": "Elm Street near the storm drain",
  "issue_type": "possible water contamination",
  "contaminants_observed": [
    "unknown"
  ],
  "urgency": "HIGH",
  "affected_entities": [
    "creek",
    "residents"
  ],
  "department": "Environmental Services",
  "resident_language": "English"
}


## 5. Test cases

These examples cover multiple environmental scenarios and one non-English report. The model output should be reviewed against the source text rather than assumed to be correct.


In [ ]:
test_messages = [
    {
        "label": "Pipe leak",
        "message": "The water main on Maple Avenue has been leaking for several days, causing a large puddle and wasting water."
    },
    {
        "label": "Creek dumping",
        "message": "There is a large amount of trash near the creek behind the shopping center on Capitol Avenue."
    },
    {
        "label": "Cloudy tap water",
        "message": "My tap water looks cloudy and tastes metallic this morning. I live near Pine Street."
    },
    {
        "label": "Blocked storm drain in Spanish",
        "message": "La alcantarilla en Oak Street y King Avenue está bloqueada con basura después de la lluvia y el agua se está desbordando."
    },
]

text_results = []
for item in test_messages:
    print(f"=== {item['label']} ===")
    print(f"Input: {item['message']}")
    try:
        result, errors = extract_structured(item["message"])
        text_results.append({"label": item["label"], "result": result, "errors": errors})
        print(json.dumps(result, indent=2, ensure_ascii=False))
        print("Validation:", "Valid" if not errors else errors)
    except Exception as exc:
        print("Request or parsing error:", exc)
    print()


## 6. Image analysis

The image workflow accepts an uploaded image and asks the model to return the same structured report format. Image analysis can identify visible conditions, but it cannot confirm contamination, identify hidden hazards, or replace an official inspection.
**Portfolio note:** Use a synthetic or public-domain image when sharing the notebook. Do not upload private resident images or personally identifying information.


In [ ]:
uploaded = files.upload()
image_filename = next(iter(uploaded))

print("Uploaded:", image_filename)

In [ ]:
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


def analyze_image(image_path, question):
    """
    Analyze an uploaded image using the OpenAI API.
    Returns the model's text response.
    """

    base64_image = encode_image(image_path)

    mime_type, _ = mimetypes.guess_type(image_path)
    mime_type = mime_type or "image/png"

    image_data_url = f"data:{mime_type};base64,{base64_image}"

    response = client.responses.create(
        model=MODEL_NAME,
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": f"""
You are analyzing an environmental image for a preliminary civic report.

Question:
{question}

Instructions:
- Describe only what is visibly supported by the image.
- Do not claim that water is safe or unsafe.
- Do not identify private individuals.
- Clearly explain uncertainty.
- This is a preliminary assessment and should be reviewed by a person.
"""
                    },
                    {
                        "type": "input_image",
                        "image_url": image_data_url,
                        "detail": "auto"
                    }
                ]
            }
        ]
    )

    return response.output_text

In [ ]:
question = """
Describe the environmental problem visible in this image.
Explain the possible community or public-safety impact.
Rate the urgency as LOW, MEDIUM, or HIGH.
Recommend which city department should review the issue.
"""

answer, usage = analyze_image(image_filename, question)

print("--- IMAGE ANALYSIS ---")
print(answer)

print("\n--- TOKEN USAGE ---")
print(getattr(usage, "total_token_count", "Not available"))


In [ ]:
civic_questions = [
    (
        "PROBLEM",
        "Describe the civic or environmental problem visible in this image. "
        "Be specific about what you see."
    ),
    (
        "IMPACT",
        "What possible public health, safety, infrastructure, or community impacts "
        "could result from what is shown?"
    ),
    (
        "URGENCY",
        "Rate the situation as LOW, MEDIUM, or HIGH urgency. "
        "Explain your rating in 2–3 sentences."
    ),
    (
        "ACTION",
        "Which city department or service may need to review this issue? "
        "What preliminary action could they consider?"
    )
]

civic_results = {
    "image_filename": image_filename,
    "answers": {}
}

for label, question in civic_questions:
    print(f"--- {label} ---")

    try:
        answer = analyze_image(image_filename, question)
        civic_results["answers"][label] = answer
        print(answer)

    except Exception as error:
        print("Analysis failed:")
        print(error)

    print()


## 7. Failure case: unstructured output

A prompt that does not specify a schema may return natural language instead of JSON. That output may be understandable to a person but difficult for an automated system to parse.


In [ ]:
failure_prompt = "Extract water issue info."
failure_example = (
    "There's dirty water and trash near a creek on Elm Street. "
    "This appears to be a water contamination issue with possible waste pollutants."
)

print("Prompt:", failure_prompt)
print("Unstructured output:")
print(failure_example)

try:
    json.loads(failure_example)
except json.JSONDecodeError as exc:
    print("\nExpected parsing failure:", exc)
    print("\nLesson: a schema-guided prompt and response validation are needed before storing the result.")


## 8. Evaluation plan

The current notebook demonstrates the workflow and validation logic. It does not make formal accuracy claims. To evaluate the system, compare model outputs with a labeled test set.

Recommended metrics include:

- JSON validity rate
- Issue-type accuracy
- Urgency accuracy
- Department-suggestion accuracy
- Language-detection accuracy
- Location-extraction accuracy
- Human-review rate
- Common error categories

For high-urgency, low-confidence, or ambiguous reports, human review should be required before escalation.


In [ ]:
# Example evaluation template. Fill these values after reviewing model outputs.
evaluation_summary = {
    "number_of_test_reports": len(test_messages),
    "json_validity_rate": "not measured yet",
    "issue_type_accuracy": "not measured yet",
    "urgency_accuracy": "not measured yet",
    "department_accuracy": "not measured yet",
    "notes": "A larger labeled dataset is needed before reporting formal performance."
}

print(json.dumps(evaluation_summary, indent=2))


## 9. Oversight, limitations, and responsible use

### Human oversight

Reports labeled HIGH or CRITICAL, reports with unclear locations, and reports with low-confidence or ambiguous outputs should be reviewed by a person before any escalation.

### Limitations

- The model may misclassify a report or image.
- An image cannot confirm chemical or biological contamination.
- The prototype does not connect to live city systems, IoT sensors, real-time alerts, or official emergency services.
- Department suggestions are preliminary and should be verified through current official procedures.
- Sample reports are synthetic.

### Responsible use

This system is intended to organize reports and support preliminary triage. It should not determine whether water is safe to drink, make emergency decisions independently, replace city inspections, or assign blame to a person or community.


## 10. Future improvements

- Add confidence scores and automatic human-review flags.
- Build a Streamlit interface.
- Expand the labeled evaluation dataset.
- Store structured reports in a relational database.
- Create a Power BI dashboard showing issue volume by location, category, urgency, and date.
- Test multilingual performance with a balanced dataset.
- Integrate official reporting channels only after validation, privacy review, and authorization.

## Conclusion

This prototype demonstrates how schema-guided multimodal AI can convert inconsistent environmental reports into structured records that are easier to review and analyze. Its strongest contribution is the combination of structured output, failure-case testing, image analysis, validation, and human oversight.
